In [ ]:


import numpy as np 
import pandas as pd 



import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



import kagglehub


/kaggle/input/competitions/spaceship-titanic/sample_submission.csv
/kaggle/input/competitions/spaceship-titanic/train.csv
/kaggle/input/competitions/spaceship-titanic/test.csv


In [2]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load datasets
train = pd.read_csv("/kaggle/input/competitions/spaceship-titanic/train.csv")
test = pd.read_csv("/kaggle/input/competitions/spaceship-titanic/test.csv")

print("Training data shape:", train.shape)
print("Testing data shape:", test.shape)

print(train.head())


Training data shape: (8693, 14)
Testing data shape: (4277, 13)
  PassengerId HomePlanet CryoSleep  Cabin  Destination   Age    VIP  \
0     0001_01     Europa     False  B/0/P  TRAPPIST-1e  39.0  False   
1     0002_01      Earth     False  F/0/S  TRAPPIST-1e  24.0  False   
2     0003_01     Europa     False  A/0/S  TRAPPIST-1e  58.0   True   
3     0003_02     Europa     False  A/0/S  TRAPPIST-1e  33.0  False   
4     0004_01      Earth     False  F/1/S  TRAPPIST-1e  16.0  False   

   RoomService  FoodCourt  ShoppingMall     Spa  VRDeck               Name  \
0          0.0        0.0           0.0     0.0     0.0    Maham Ofracculy   
1        109.0        9.0          25.0   549.0    44.0       Juanna Vines   
2         43.0     3576.0           0.0  6715.0    49.0      Altark Susent   
3          0.0     1283.0         371.0  3329.0   193.0       Solam Susent   
4        303.0       70.0         151.0   565.0     2.0  Willy Santantines   

   Transported  
0        False  
1      

In [3]:

def prepare_data(df):
    df = df.copy()

    # Extract group number from PassengerId
    df["GroupId"] = df["PassengerId"].str.split("_").str[0]

    # Number of people sharing the same group
    df["GroupSize"] = df.groupby("GroupId")["PassengerId"].transform("count")

    # Split Cabin into deck, cabin number, and side
    cabin_parts = df["Cabin"].str.split("/", expand=True)

    df["CabinDeck"] = cabin_parts[0]
    df["CabinNumber"] = pd.to_numeric(
        cabin_parts[1], errors="coerce"
    )
    df["CabinSide"] = cabin_parts[2]

    # Calculate total spending
    spend_columns = [
        "RoomService", "FoodCourt", "ShoppingMall",
        "Spa", "VRDeck"
    ]

    df["TotalSpend"] = df[spend_columns].sum(
        axis=1, min_count=1
    )

    # Remove columns not used directly as model features
    df.drop(
        columns=["PassengerId", "Name", "Cabin", "GroupId"],
        inplace=True
    )

    return df


train_clean = prepare_data(train)
test_clean = prepare_data(test)

print(train_clean.head())
print("Missing values before imputation:")
print(train_clean.isnull().sum())


  HomePlanet CryoSleep  Destination   Age    VIP  RoomService  FoodCourt  \
0     Europa     False  TRAPPIST-1e  39.0  False          0.0        0.0   
1      Earth     False  TRAPPIST-1e  24.0  False        109.0        9.0   
2     Europa     False  TRAPPIST-1e  58.0   True         43.0     3576.0   
3     Europa     False  TRAPPIST-1e  33.0  False          0.0     1283.0   
4      Earth     False  TRAPPIST-1e  16.0  False        303.0       70.0   

   ShoppingMall     Spa  VRDeck  Transported  GroupSize CabinDeck  \
0           0.0     0.0     0.0        False          1         B   
1          25.0   549.0    44.0         True          1         F   
2           0.0  6715.0    49.0        False          2         A   
3         371.0  3329.0   193.0        False          2         A   
4         151.0   565.0     2.0         True          1         F   

   CabinNumber CabinSide  TotalSpend  
0          0.0         P         0.0  
1          0.0         S       736.0  
2          

In [4]:

X = train_clean.drop(columns=["Transported"])
y = train["Transported"].astype(bool)

X_test = test_clean.copy()

# Identify numerical and categorical columns
numeric_columns = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_columns = X.select_dtypes(
    include=["object", "bool", "category"]
).columns.tolist()

print("Numerical features:", numeric_columns)
print("Categorical features:", categorical_columns)


Numerical features: ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'GroupSize', 'CabinNumber', 'TotalSpend']
Categorical features: ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'CabinDeck', 'CabinSide']


In [5]:

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_columns),
        ("cat", categorical_pipeline, categorical_columns)
    ]
)


In [6]:

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

# Train the model
model.fit(X_train, y_train)

# Evaluate on validation data
val_predictions = model.predict(X_val)

accuracy = accuracy_score(y_val, val_predictions)

print(f"Validation Accuracy: {accuracy:.4f}")
print(f"Validation Accuracy: {accuracy * 100:.2f}%")


Validation Accuracy: 0.8051
Validation Accuracy: 80.51%


In [ ]:


model.fit(X, y)


test_predictions = model.predict(X_test)


submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Transported": test_predictions.astype(bool)
})

submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print(submission.head())
print("Submission rows:", len(submission))
print("Submission columns:", submission.columns.tolist())
print("File saved successfully!")


  PassengerId  Transported
0     0013_01         True
1     0018_01        False
2     0019_01         True
3     0021_01         True
4     0023_01         True
Submission rows: 4277
Submission columns: ['PassengerId', 'Transported']
File saved successfully!
